# Prueba E0 — Calibracion del delay de WPE (taps fijo en 5)

Barrido de `wpe_delay` [1,2,3] con `wpe_taps` **fijo en 5** (restriccion de memoria del FPGA)
a RT60=610 ms, para fijar `delay*` **antes** de correr las Pruebas 1-3.

Correcciones respecto de la maqueta:
- `t_early` **fijo en 8 ms** (`base_config['t_early']=0.008`), desacoplado de `wpe_delay`.
  Asi cada delay se evalua contra la MISMA referencia early y son comparables.
- `wpe_taps` fijo en 5 (no se optimiza; lo dicta el HW).
- La seleccion muestra **todas** las metricas por delay (no solo PESQ).

**Como ejecutar:** correr *Setup* una vez por sesion, luego *Ejecucion* y *Seleccion del optimo*.
Copiar el `delay*` resultante a las Pruebas 1-3 (con `taps=5`).

## Setup — ejecutar una vez por sesión de Colab
Montar Drive, clonar el repo, instalar dependencias y actualizar el código.

In [ ]:
# Import the drive module from Google Colab
from google.colab import drive

# Mount Google Drive to the virtual machine
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3. Descargar tu código temporalmente
%cd /content
!git clone https://github.com/MatiasVereert/Vision-Aided-Beamformer.git

/content
fatal: destination path 'Vision-Aided-Beamformer' already exists and is not an empty directory.


In [ ]:
# Install standard audio and evaluation libraries from PyPI
!pip install noisereduce mir_eval

# Install pysepm directly from its official GitHub repository
!pip install git+https://github.com/schmiph2/pysepm.git

# Install pb_bss directly from the Paderborn University GitHub repository
!pip install git+https://github.com/fgnt/pb_bss.git

# h5py and pyarrow are usually pre-installed, but we can ensure they are up to date
!pip install h5py pyarrow
!pip install git+https://github.com/LCAV/pyroomacoustics.git

!pip install git+https://github.com/fgnt/nara_wpe.git

!pip install paderbox

!pip install ai_edge_litert

!pip install git+https://github.com/fakufaku/fast_bss_eval.git

  Cloning https://github.com/schmiph2/pysepm.git to /tmp/pip-req-build-t349a1f_
  Running command git clone --filter=blob:none --quiet https://github.com/schmiph2/pysepm.git /tmp/pip-req-build-t349a1f_
  Resolved https://github.com/schmiph2/pysepm.git to commit 7ef88aff2c56201a2d0470aaeb58e77e47a914d2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached https://github.com/ludlows/python-pesq/archive/master.zip
  Preparing metadata (setup.py) ... done
  Using cached https://github.com/jfsantos/SRMRpy/archive/master.zip
  Preparing metadata (setup.py) ... done
  Using cached https://github.com/detly/gammatone/archive/master.zip
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/fgnt/pb_bss.git to /tmp/pip-req-build-fcp7ado4
  Running command git clone --filter=blob:none --quiet https://github.com/fgnt/pb_bss.git /tmp/pip-req-build-fcp7ado4
  Resolved https://github.com/fgnt

In [ ]:
%cd /content/Vision-Aided-Beamformer
!git pull origin main

/content/Vision-Aided-Beamformer
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 13 (delta 9), reused 13 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (13/13), 16.70 KiB | 657.00 KiB/s, done.
From https://github.com/MatiasVereert/Vision-Aided-Beamformer
 * branch            main       -> FETCH_HEAD
   17f4a7d..16236ff  main       -> origin/main
Updating 17f4a7d..16236ff
Fast-forward
 mird_benchmark_metrics.csv                         |  12 ++++++------
 src/beamforming/mask/single_dtln_mvdr_Souden.py    |   8 +++++++-
 src/evaluation/full_benchmark_test_dtln_mird.py    |   4 ++--
 .../mird_benchmark_metrics.parquet                 | Bin 171706 -> 171716 bytes
 4 files changed, 15 insertions(+), 9 deletions(-)


## Ejecución del sweep

In [ ]:
import sys
import os
import numpy as np
import shutil
from datetime import datetime


# 1. Definicion de rutas del repositorio en Colab
repo_root = '/content/Vision-Aided-Beamformer'
src_path = os.path.join(repo_root, 'src')

if repo_root not in sys.path:
    sys.path.append(repo_root)
if src_path not in sys.path:
    sys.path.append(src_path)

%cd {src_path}

try:
    import tensorflow as tf
    TFLITE_AVAILABLE = True
except ImportError:
    print("[!] TensorFlow no detectado. Los modulos neuronales DTLN se desactivaran.")
    TFLITE_AVAILABLE = False

# Orquestador + wrapper del sistema (nombres NUEVOS del bf_wrappers local).
# E0 solo necesita el sistema NM-MVDR.
from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from evaluation.bf_wrappers import NM_MVDR
from propagation.mird_loader import MirdDatasetProvider, generate_mird_linear_array

# 2. Inicializacion de los interpretes DTLN
model_1_path = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite")
model_2_path = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_2.tflite")

interpreter_1, interpreter_2 = None, None
if TFLITE_AVAILABLE and os.path.exists(model_1_path) and os.path.exists(model_2_path):
    try:
        interpreter_1 = tf.lite.Interpreter(model_path=model_1_path)
        interpreter_1.allocate_tensors()
        interpreter_2 = tf.lite.Interpreter(model_path=model_2_path)
        interpreter_2.allocate_tensors()
        print("[*] Modelos DTLN TFLite cargados correctamente.")
    except Exception as e:
        print(f"[!] Fallo la reserva de memoria para DTLN: {e}. Se procedera sin mejora neuronal.")
        interpreter_1, interpreter_2 = None, None
else:
    print("[*] Ejecutando en modo estrictamente Acustico/Espacial (Sin DTLN).")

input_dir = "/content/drive/MyDrive/Benchmarks_tesis/inputs"
mird_dir  = "/content/drive/MyDrive/Benchmarks_tesis/rirs"
provider = MirdDatasetProvider(root_dir=mird_dir)

base_config = {
    'fs': 16000,
    'duration': 20,
    't_early': 0.008,               # <-- FIJO en 8 ms (desacoplado de wpe_delay)
    'array_center': [3.0, 3.0, 1.2],
    'mird_spacing': "3-3-3-8-3-3-3",

    'snr_db': 60.0,
    'source_path': os.path.join(input_dir, "p002_emo_adoration_sentences.wav"),
    'interf_paths': [
        os.path.join(input_dir, "hairdryer_07_SH_MKH800.wav"),
        os.path.join(input_dir, "flute_music.wav"),
    ],

    # Fallbacks (los sobreescribe el param_grid). taps FIJO en 5.
    'wpe_taps': 5,
    'wpe_delay': 1,
    'wpe_alpha': 0.9999,
    'wpe_stft_size': 512,
    'wpe_stft_shift': 128,

    'stft_window': 512,
    'stft_overlap': 384,

    'dtln_model_path': os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite"),
    'eval_references': ['anechoic', 'early', 'reverberant'],
}

# Escena de estres (RT=610, iSIR=0, 1-2 interferentes). Solo se BARRE el delay.
param_grid = {
    'rt60': [0.610],
    'target_angle': [0],
    'target_dist': [1.0],
    'interf_configs': [
        [(45, 1.0)],
        [(-90, 1.0)],
        [(45, 1.0), (-90, 1.0)],
    ],
    'isir_db': [0],
    'mismatch_gain': [0],
    'mismatch_phase': [0],
    'use_wpe': [True],
    'wpe_taps':  [5],          # <-- FIJO (restriccion HW)
    'wpe_delay': [1, 2, 3],    # <-- lo que se calibra
    'error_angle_deg': [0.0],
    'error_distance_m': [0.0],
}

# Sistema NM-MVDR (mascara DTLN + Souden MVDR), config del sistema desplegado.
processors_dict = {
    "NM-MVDR": NM_MVDR(min_loading=1e-6, alpha=0.99),
}


# --- EJECUCION ---
print("\n" + "="*60)
print("INICIANDO CALIBRACION DE DELAY DE WPE (taps=5 fijo)")
print("="*60)

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M")
temp_output_dir  = f"/content/results_temp/wpe_E0_delaysweep_{RUN_TAG}"
drive_output_dir = f"/content/drive/MyDrive/Tesis_Beamformers/results/wpe_E0_delaysweep_{RUN_TAG}"

os.makedirs(temp_output_dir, exist_ok=True)
os.makedirs(drive_output_dir, exist_ok=True)

df_E0 = run_mird_grid_search(
    grid_params=param_grid,
    dataset_provider=provider,
    processors=processors_dict,
    scene_base_config=base_config,
    output_dir=temp_output_dir,
    interpreter_1=interpreter_1,
    interpreter_2=interpreter_2,
)

print("\n[INFO] Procesamiento finalizado. Sincronizando a Google Drive...")
shutil.copytree(temp_output_dir, drive_output_dir, dirs_exist_ok=True)
print("\n[EXITO] E0 completado y respaldado en Google Drive.")

## Selección del óptimo

In [ ]:
# --- Seleccion del delay* sobre NM-MVDR: TODAS las metricas ---
import pandas as pd
import numpy as np

df = pd.read_csv(os.path.join(drive_output_dir, "mird_benchmark_metrics.csv"))
df = df[df["processor"] == "NM-MVDR"].copy()

# Metricas end-to-end (Delta_tot). PESQ tambien vs anecoica (convencion).
# CD: MENOR es mejor. El resto: MAYOR es mejor.
metric_specs = [
    ("Delta_tot_PESQ_anechoic", "PESQ(anec)", "up"),
    ("Delta_tot_PESQ_early",    "PESQ(early)", "up"),
    ("Delta_tot_STOI_early",    "STOI", "up"),
    ("Delta_tot_SDR_early",     "SDR",  "up"),
    ("Delta_tot_SIR_early",     "SIR",  "up"),
    ("Delta_tot_SAR_early",     "SAR",  "up"),
    ("Delta_tot_SINR_early",    "SINR", "up"),
    ("Delta_tot_CD_early",      "CD",   "down"),
]
present = [(c, lbl, d) for c, lbl, d in metric_specs if c in df.columns]
cols = [c for c, _, _ in present]

# taps esta fijo en 5 -> agregamos por delay (media sobre las 3 escenas de interferencia).
tab = df.groupby("wpe_delay")[cols].mean()
tab.columns = [lbl for _, lbl, _ in present]
print("=== E0 | NM-MVDR end-to-end por wpe_delay (taps=5) | media sobre escenas ===")
print("Direccion:", "  ".join(f"{lbl}[{'^' if d=='up' else 'v'}]" for _, lbl, d in present))
print(tab.round(3).to_string())

# Aporte del WPE por si solo (dereverberacion pura), para ver como cambia con el delay.
wpe_specs = [("Delta_wpe_PESQ_early","PESQ"), ("Delta_wpe_STOI_early","STOI"),
             ("Delta_wpe_SDR_early","SDR"), ("Delta_wpe_CD_early","CD")]
wpe_present = [(c, lbl) for c, lbl in wpe_specs if c in df.columns]
if wpe_present:
    wcols = [c for c, _ in wpe_present]
    tabw = df.groupby("wpe_delay")[wcols].mean()
    tabw.columns = [lbl for _, lbl in wpe_present]
    print("\n=== Aporte de WPE solo (Delta_wpe, vs early) por delay ===")
    print(tabw.round(3).to_string())

print("\n>>> Elegi delay* mirando TODAS las metricas.")
print(">>> Un delay muy bajo puede sobre-blanquear (sube SDR/SIR pero baja PESQ / sube CD).")
print(">>> Copia delay* a las Pruebas 1-3 (con taps=5 fijo).")